# Train Pneumonia Detection Model (Run on Google Colab with free GPU)
Runtime > Change runtime type > GPU, before running cells.

In [ ]:
# Step 1: Install Kaggle and set up API token
!pip install -q kaggle

# Paste the token you copied from Kaggle (the one starting with KGAT_...)
KAGGLE_API_TOKEN = 'KGAT_05d349fc2d8e81ae67e3c96fb6a8ac16'  # <-- replace this

import os
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
with open(os.path.expanduser('~/.kaggle/access_token'), 'w') as f:
    f.write(KAGGLE_API_TOKEN)
!chmod 600 ~/.kaggle/access_token

!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia
!unzip -q chest-xray-pneumonia.zip -d data


In [ ]:
DATA_DIR = 'data/chest_xray'
import os
print(os.listdir(DATA_DIR))


In [ ]:
# Step 2: Download the train.py script logic inline (copy from model/train.py)
# Or upload train.py and run: !python train.py --data_dir data/chest_xray --epochs 10
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(*IMG_SIZE, 3))
base_model.trainable = False
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.2)(x)
output = Dense(1, activation='sigmoid')(x)
model = Model(inputs=base_model.input, outputs=output)
model.compile(optimizer=Adam(1e-4), loss='binary_crossentropy', metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])
model.summary()


In [ ]:
train_datagen = ImageDataGenerator(rescale=1./255, rotation_range=15, width_shift_range=0.1, height_shift_range=0.1, zoom_range=0.1)
val_test_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(f'{DATA_DIR}/train', target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='binary', classes=['NORMAL','PNEUMONIA'])
val_gen = val_test_datagen.flow_from_directory(f'{DATA_DIR}/val', target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='binary', classes=['NORMAL','PNEUMONIA'])
test_gen = val_test_datagen.flow_from_directory(f'{DATA_DIR}/test', target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='binary', classes=['NORMAL','PNEUMONIA'], shuffle=False)

callbacks = [
    ModelCheckpoint('pneumonia_model.h5', monitor='val_auc', mode='max', save_best_only=True, verbose=1),
    EarlyStopping(monitor='val_auc', mode='max', patience=4, restore_best_weights=True)
]

history = model.fit(train_gen, validation_data=val_gen, epochs=10, callbacks=callbacks)


In [ ]:
results = model.evaluate(test_gen)
print(dict(zip(model.metrics_names, results)))
model.save('pneumonia_model.h5')
from google.colab import files
files.download('pneumonia_model.h5')
